In [ ]:
import abc
import contextlib
import os
from collections.abc import Sequence
from functools import cached_property
from itertools import accumulate
from pathlib import Path
from typing import Any, Iterator, Literal, Protocol

import equinox as eqx
import fsspec
import grain
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import xarray as xr
import yaml
import zarr
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
    ContextAppendedFluxNO,
    DISCO,
    DPOT,
    HyperFluxFNOLocal,
    NaiveFluxNO,
    NaiveFNO,
    SpatiotemporalFNO,
)
from context_flux_no.data.sources import WellDatasetSourceBase,HierarchicalStorage,ZarrWellDatasetSource
from context_flux_no.training import PDEDataset
from context_flux_no.training.loader import (
    SegmentLoader,
    SegmentLoaderBackground,
)
from context_flux_no.training.trainer import Trainer
from einops import pack, rearrange
from jaxtyping import Array, Float, PRNGKeyArray


jax.config.update("jax_default_device", jax.devices("gpu")[1])

In [ ]:
source = ZarrWellDatasetSource(
    "../../data/datasets",
    "cubic_1d",
    "train",
    window_size=21,
    restrict_trajectory_lengths_to=80,
    preload_into_ram=True
)
source.datapaths

In [ ]:
source[0]

In [ ]:
sampler = grain.samplers.IndexSampler(len(source), shuffle=True, seed=0)
transforms = [grain.transforms.Batch(batch_size=128)]
dataloader = grain.DataLoader(
    data_source=source,
    sampler=sampler,
    operations=transforms,
    worker_count=0,
    # read_options=grain.ReadOptions(num_threads=32, prefetch_buffer_size=1000),
)

In [ ]:
dataiter = iter(dataloader)
batch = next(dataiter)

In [ ]:
batch.shape

In [ ]:
source.metadata_common["temporal_resolution"]

In [ ]:
batch.shape

In [ ]:
batch = next(dataiter)

In [ ]:
fluxno_appended = ContextAppendedFluxNO(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    depth=5,
    lift_dim=128,
    stencil_size=(10, 10),
    width_hyper=512,
    depth_hyper=3,
    width_flux=512,
    key=jax.random.key(0),
)
print(fluxno_appended.num_parameters() / 1e6)

In [ ]:
fluxno_naive = NaiveFluxNO(
    num_spatial_dims=1,
    channels=1,
    lift_dim=512,
    depth=6,
    stencil_widths=(10, 10),
    hidden_dim=512,
    key=jax.random.key(0),
)

fluxno_naive.num_parameters() / 1e6

In [ ]:
fno_naive = NaiveFNO(
    num_spatial_dims=1,
    channels=1,
    lift_dim=128,
    depth=10,
    frequency_modes=12,
    key=jax.random.key(0),
)

fno_naive.num_parameters() / 1e6

In [ ]:
fno = SpatiotemporalFNO(
    num_spatial_dims=1,
    channels=1,
    lift_dim=128,
    depth=2,
    frequency_modes=(3, 12),
    key=jax.random.key(0),
)
fno.num_parameters() / 1e6

In [ ]:
dpot = DPOT(
    num_spatial_dims=1,
    in_channels=1,
    out_channels=1,
    in_timesteps=20,
    grid_size=(100,),
    patch_size=(4,),
    embedding_dim=128,
    max_frequency_modes=12,
    fno_depth=10,
    num_blocks=8,
    hidden_dim_patch=512,
    hidden_dim_fno=512,
    hidden_dim_output=512,
    key=jax.random.key(0),
)
print(dpot.num_parameters() / 1e6)

In [ ]:
hyperfluxfno = HyperFluxFNO(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=64,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=64,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    depth=4,
    frequency_modes=12,
    lift_dim=128,
    stencil_size=(3, 3),
    width_lift=128,
    width_project=128,
    width_hyper=128,
    blocks_hyper=2,
    blocks_flux=4,
    key=jax.random.key(0),
)
print(hyperfluxfno.num_parameters() / 1e6)

In [ ]:
hyperfluxfno = HyperFluxFNOLocal(
    num_spatial_dims=1,
    in_channels=1,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    depth=4,
    lift_dim=128,
    stencil_size=(10, 10),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxfno.num_parameters() / 1e6)

In [ ]:
# @partial(eqx.filter_grad, has_aux=True)
@eqx.filter_jit
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


test_data = jax.random.normal(jax.random.key(0), shape=(512, 21, 1, 100))

In [ ]:
test_data[0, :20].shape

In [ ]:
# print(test_data.size * test_data.itemsize / 1e9)
# eqx.filter_jit(eqx.filter_grad(loss_fn, has_aux=True))(

# )
compiled = (
    eqx.filter_jit(lambda model, x, args, key: model(x, args, key=key))
    .lower(hyperfluxfno, test_data[0, :-1], (1.0, 1.0), jax.random.key(0))
    .compile()
)
flops = compiled.compiled.cost_analysis()["flops"] / 1e9
print(flops)  # 2.0

In [ ]:
forward = eqx.filter_jit(lambda model, x, args, key: model(x, args, key=key))

In [ ]:
%%timeit
forward(disco, test_data[0, :-1], (1.0, 1.0), jax.random.key(0))[0].block_until_ready()

In [ ]:
compiled.compiled.cost_analysis()["flops"] / 1e9

In [ ]:
disco = DISCO(
    num_spatial_dims=1,
    channels=1,
    embedding_dim=128,
    patch_size=4,
    num_hypernet_blocks=3,
    droppath=0.1,
    num_hypernet_heads=8,
    mlp_hidden_dim=32,
    boundary_condition="periodic",
    opnet_hidden_channels=4,
    opnet_norm_groups=2,
    rtol=1e-4,
    atol=1e-6,
    max_steps=16,
    key=jax.random.key(0),
)

In [ ]:
print(disco.num_parameters() / 1e6)

In [ ]:
hyperfluxfno.hypernetwork_heads[0]

In [ ]:
dataloader = SegmentLoaderBackground(
    dataset_xr, segment_length=21, batch_size=1024, batches_per_load=50
)
dataiter = iter(dataloader)

In [ ]:
arr = next(dataiter)
(arr.dtype.itemsize * arr.size) * 5 / (1024 * 1024)

In [ ]:
arr.shape

In [ ]:
# LossFunction[M: eqx.Module, B: PyTree] = Callable[[M, B, Any, PRNGKeyArray], FloatScalar]

In [ ]:
hyperfluxfno

In [ ]:
loss_fn(fluxno_appended, arr, (1.0, 1.0), jax.random.key(0))

In [ ]:
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


trainer = Trainer(
    optax.adamw(1e-3),
    loss_fn,
    "./",
    None,
    {"entity": "jhko725", "project": "hyperfluxfno"},
)

In [ ]:
trainer.train(dpot, dataloader, None, None, num_steps=500, seed=0)

In [ ]:
rng = np.random.default_rng(0)
inds_traj = rng.integers(0, 10000, size=(40, 1024))
inds_t0 = rng.integers(0, rolled.sizes["t"] - 20, size=(40, 1024))


def load_single(dataset, idx_traj, idx_t0, segment_length):
    return dataset.isel(
        {
            "traj": idx_traj,
            "t": idx_t0 + segment_length - 1,
        }
    )


load_single(
    rolled["values"],  # .chunk({"x": 5, "segment": 1, "traj": -1, "t": -1}),
    xr.DataArray(inds_traj, dims=["metabatch", "batch"]),
    xr.DataArray(inds_t0, dims=["metabatch", "batch"]),
    21,
)  # .compute().to_numpy()

In [ ]:
dataset_train, dataset_test = PDEDataset.from_xarray(dataset_xr).split_by_time(80)

In [ ]:
dataset_xr.isel({"t": slice(None, None, 10)})

In [ ]:
dataset_train

In [ ]:
loader = SegmentLoader(
    dataset_train,
    segment_length=21,
    batch_size=50,
    batching_strategy="consecutive_segments",
)
batch, _ = loader.load_batch(loader.init())

In [ ]:
batch[0].shape

In [ ]:
dpot = DPOT(
    num_spatial_dims=1,
    in_channels=1,
    out_channels=1,
    in_timesteps=20,
    patch_size=(4,),
    img_size=(100,),
    embedding_dim=256,
    max_frequency_modes=(10,),
    fno_depth=5,
    num_blocks=4,
    num_classes=12,
    hidden_dim_patch=256,
    hidden_dim_fno=256,
    hidden_dim_output=32,
    key=jax.random.key(0),
)
dpot.num_parameters()  # 0.238M

In [ ]:
disco = DISCO(
    num_spatial_dims=1,
    channels=1,
    embedding_dim=48,
    patch_size=100,
    num_hypernet_blocks=2,
    droppath=0.1,
    num_hypernet_heads=2,
    mlp_hidden_dim=48,
    boundary_condition="periodic",
    key=jax.random.key(0),
)
disco.num_parameters()  # 0.807M

In [ ]:
hyperfluxfno = HyperFluxFNO(
    data_dim=1,  # Rename argument to channels
    depth=4,
    frequency_modes=10,
    lift_dim=48,
    context_embed_dim=48,
    stencil_size=(2, 1),
    patch_size=(5, 25),
    hypernet_init="bias-hyperinit",
    num_hidden_patch=1,
    key=jax.random.key(0),
)
hyperfluxfno.num_parameters()  # 0.577M

In [ ]:
hyperfluxfno = HyperFluxFNOv1(
    data_dim=1,  # Rename argument to channels
    depth=4,
    frequency_modes=7,
    lift_dim=48,
    context_embed_dim=48,
    stencil_size=(7, 7),
    patch_size=(5, 25),
    hypernet_init="bias-hyperinit",
    num_hidden_patch=1,
    key=jax.random.key(0),
)
hyperfluxfno.num_parameters()  # 0.438M

In [ ]:
dx = dataset_train.x[1] - dataset_train.x[0]
dt = dataset_train.t[1] - dataset_train.t[0]
dt

In [ ]:
eqx.filter_value_and_grad(loss_fn, has_aux=True)(
    hyperfluxfno, batch, (dt, dx), jax.random.key(0)
)

## Evaluate training results

In [ ]:
loader_test = SegmentLoader(dataset_test, segment_length=21, batch_size=1024)

batch_test, _ = loader_test.load_batch(loader_test.init())
model: AbstractMultiphysicsOperator = eqx.nn.inference_mode(model)
u1_pred = eqx.filter_vmap(lambda u_: model(u_, (dt, dx), key=jax.random.key(0)))(
    batch_test[0][:, :-1]
)[0]
u1_data = batch_test[0][:, -1]

In [ ]:
sample_idx = 10

fig, axes = plt.subplots(2, 1, figsize=(5, 3), sharex=True, height_ratios=(2, 1))
axes[0].plot(dataset_test.x, u1_data[sample_idx, 0], label="Data")
axes[0].plot(dataset_test.x, u1_pred[sample_idx, 0], label="Pred")

axes[1].plot(dataset_test.x, u1_pred[sample_idx, 0] - u1_data[sample_idx, 0])

In [ ]:
fig, ax

In [ ]:
u_rollout_pred = eqx.filter_vmap(
    lambda u_: model.rollout(u_, (dt, dx), num_steps=20, key=jax.random.key(0))
)(batch_test[0][:, :-1])[0]

In [ ]:
u_rollout_pred.shape

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(5, 3), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(dataset_test.x, u_rollout_pred[sample_idx, 5 * i, 0])